# Highway Network na CIFAR-10

[Highway Networks](https://arxiv.org/abs/1505.00387) (Srivastava, Greff, Schmidhuber, 2015)

In [ ]:
import io, os, time, urllib.request
import numpy as np
import pyarrow.parquet as pq
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
EPOCHS = 100
BATCH_SIZE = 128
LR_DECAY = 0.1
LR_MILESTONES = (0.5, 0.75)

SEARCH_TRIALS = 20
SEARCH_EPOCHS = 5
SEARCH_VAL = 5000
LR_RANGE = (0.01, 0.2)
MOMENTUM_RANGE = (0.8, 0.95)
GATE_BIAS_RANGE = (-4.0, -1.0)

SEED = 0

torch.manual_seed(SEED)
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
print("GPU:", torch.cuda.get_device_name(0))

## Dane

CIFAR-10: 50 000 obrazów treningowych i 10 000 testowych, 32x32 RGB, 10 klas.

In [ ]:
def load_split(split):
    url = f"https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/{split}-00000-of-00001.parquet"
    path = f"cifar10_{split}.parquet"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    table = pq.read_table(path).to_pydict()
    images = np.stack([np.asarray(Image.open(io.BytesIO(im["bytes"])).convert("RGB")) for im in table["img"]])
    labels = np.array(table["label"])
    return images, labels

x_train, y_train = load_split("train")
x_test, y_test = load_split("test")
print(x_train.shape, x_test.shape)

In [ ]:
x_train = torch.tensor(x_train).permute(0, 3, 1, 2).float() / 255  # [N, 32, 32, 3] -> [N, 3, 32, 32], wartości 0-1
x_test = torch.tensor(x_test).permute(0, 3, 1, 2).float() / 255
# normalizacja każdego kanału RGB: inicjalizacja He zakłada wejście o średniej 0 i wariancji 1
mean = x_train.mean(dim=(0, 2, 3), keepdim=True)
std = x_train.std(dim=(0, 2, 3), keepdim=True)
x_train, x_test = (x_train - mean) / std, (x_test - mean) / std
y_train, y_test = torch.tensor(y_train), torch.tensor(y_test)

# augmentacja jak w paperze ("random translations"): losowe przesunięcie o maks. 4 piksele w każdą stronę,
# więc w każdej epoce sieć widzi trochę inne obrazy i trudniej jej zapamiętać zbiór treningowy
shift = T.RandomCrop(32, padding=4)

class Shifted(torch.utils.data.Dataset):
    def __init__(self, x, y): self.x, self.y = x, y
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return shift(self.x[i]), self.y[i]

def loader(x, y, train):
    ds = Shifted(x, y) if train else torch.utils.data.TensorDataset(x, y)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=train, drop_last=train,
                                       num_workers=2, pin_memory=True, persistent_workers=True)

## Warstwa highway

$$y = H(x)\cdot T(x) + x\cdot\big(1 - T(x)\big)$$

$$H(x) = \mathrm{ReLU}(W_H * x + b_H), \qquad T(x) = \sigma(W_T * x + b_T)$$

$T(x)$ [1] decyduje, jaka część wyjścia to przetworzone $H(x)$, a jaka niezmienione $x$ [2]. $x$ i $H(x)$ muszą mieć ten sam rozmiar [3].

[1] $*$ to konwolucja: filtr 3x3 przesuwa się po obrazie i w każdym miejscu mnoży piksele pod sobą przez swoje wagi i sumuje (po wszystkich kanałach), co daje jeden piksel wyjścia. $\sigma$ to sigmoida, więc $T(x)$ ma wartości od 0 do 1, osobno dla każdego kanału i piksela.

[2] Dla $T = 0$ warstwa przepuszcza $x$ bez zmian, razem z gradientem. Dzięki temu gradient nie zanika nawet w głębokiej sieci.

[3] Stąd padding 1: ramka z zer wokół obrazu sprawia, że filtr 3x3 daje wyjście tego samego rozmiaru co wejście. Liczbę kanałów zmienia tylko zwykła warstwa `PlainConv2d`.

In [ ]:
class PlainConv2d(nn.Module):  # zwykła warstwa y = H(x), zmienia liczbę kanałów
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.H = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        nn.init.kaiming_normal_(self.H.weight, nonlinearity="relu")
        nn.init.zeros_(self.H.bias)

    def forward(self, x):
        return F.relu(self.H(x))


class HighwayConv2d(nn.Module):
    def __init__(self, channels, gate_bias):
        super().__init__()
        # filtr 3x3 jak w Fitnet 4, na której paper opiera sieć; padding=1 zachowuje rozmiar obrazu
        self.H = nn.Conv2d(channels, channels, 3, padding=1)  # W_H, b_H
        self.T = nn.Conv2d(channels, channels, 3, padding=1)  # W_T, b_T
        # inicjalizacja He (jak w paperze dla sieci na MNIST): wagi z rozkładu normalnego o wariancji 2/n
        # (n = liczba wejść neuronu); przy ReLU sygnał ma taką samą wariancję w kolejnych warstwach
        nn.init.kaiming_normal_(self.H.weight, nonlinearity="relu")
        nn.init.kaiming_normal_(self.T.weight, nonlinearity="relu")
        nn.init.zeros_(self.H.bias)
        # ujemny b_T: na starcie T bliskie 0, więc warstwy głównie przepuszczają wejście
        nn.init.constant_(self.T.bias, gate_bias)

    def forward(self, x):
        h = F.relu(self.H(x))
        t = torch.sigmoid(self.T(x))
        return h * t + x * (1 - t)

## Sieć

19 warstw i około 2,3M parametrów, jak sieć, która w paperze osiąga 92,24% na CIFAR-10.

```
wejście 3x32x32
-> Plain(3->32)   + 5 x Highway(32)  -> MaxPool -> 32x16x16
-> Plain(32->80)  + 5 x Highway(80)  -> MaxPool -> 80x8x8
-> Plain(80->128) + 5 x Highway(128)            -> 128x8x8
-> średnia po 8x8 -> Linear(128->10)
```

In [ ]:
class HighwayNet(nn.Module):
    def __init__(self, gate_bias):
        super().__init__()
        layers, channels = [], 3
        for i, width in enumerate((32, 80, 128)):  # szerokości dobrane tak, żeby wyszło ok. 2,3M parametrów
            layers.append(PlainConv2d(channels, width))
            layers += [HighwayConv2d(width, gate_bias) for _ in range(5)]
            if i < 2:
                # zmniejszenie obrazu: z każdego bloku 2x2 pikseli zostaje maksimum, 32x32 -> 16x16 -> 8x8;
                # dzięki temu liczba operacji (ok. 350M mnożeń na obraz) jest porównywalna z Fitnet 4
                layers.append(nn.MaxPool2d(2))
            channels = width
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Linear(channels, 10)
        nn.init.kaiming_normal_(self.classifier.weight, nonlinearity="linear")
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        x = self.features(x).mean(dim=(2, 3))  # średnia każdej mapy cech: [N, 128, 8, 8] -> [N, 128]
        return self.classifier(x)  # 10 logitów


net = HighwayNet(gate_bias=-2.0)
print("parametry:", sum(p.numel() for p in net.parameters()))

## Trening

Zwykłe SGD z momentum, w jednym etapie, bez sieci-nauczyciela.

In [ ]:
def run_epoch(net, loader, opt=None, scaler=None):
    train = opt is not None
    net.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):  # fp16
                logits = net(x)
            loss = F.cross_entropy(logits.float(), y)  # softmax i -log p poprawnej klasy
            if train:
                opt.zero_grad()
                # GradScaler mnoży stratę przed liczeniem gradientów, żeby małe gradienty nie zaokrągliły się w fp16 do zera
                scaler.scale(loss).backward()
                scaler.step(opt)  # krok SGD na gradientach podzielonych z powrotem przez tę samą liczbę
                scaler.update()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n


def fit(lr, gate_bias, momentum, epochs, train_loader, val_loader=None, verbose=True):
    torch.manual_seed(SEED)  # ta sama inicjalizacja i kolejność danych dla każdego ustawienia
    net = HighwayNet(gate_bias).to(DEVICE)
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=momentum)
    milestones = [int(epochs * m) for m in LR_MILESTONES]
    # lr x0,1 po 50% i 75% epok: duży lr szybko znajduje dobry obszar, mały dopracowuje wagi
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones, gamma=LR_DECAY)
    scaler = torch.amp.GradScaler()
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(net, train_loader, opt, scaler)
        rec = {"train_loss": train_loss, "train_acc": train_acc}
        msg = f"epoka {epoch:3d}/{epochs} | train loss {train_loss:.3f}, acc {train_acc:.3f}"
        if val_loader is not None:
            rec["val_loss"], rec["val_acc"] = run_epoch(net, val_loader)
            msg += f" | val loss {rec['val_loss']:.3f}, acc {rec['val_acc']:.3f}"
        history.append(rec)
        sched.step()
        if verbose:
            print(f"{msg} | {time.time() - t0:.0f} s")
    return net, history